# Project 23 — Depth-First Search: Enterprise Graph Explorer

**Business problem:** deeply traverse a large relationship graph to discover reachable entities, connected components, cycles and hidden dependency chains.

This project uses the **Stanford SNAP Email-Eu-core network**, representing email relationships inside a European research institution. It implements iterative DFS from scratch, validates results with NetworkX, explores component structure, detects graph cycles, visualises a real traversal and benchmarks search performance.

**Big-company connection:** DFS is a core graph primitive behind dependency analysis, repository traversal, identity/security graph investigations and infrastructure topology exploration. A GitHub-style dependency graph, for example, can be traversed recursively to discover everything reachable from a package or repository.

**Algorithmic complexity:** DFS runs in **O(V + E)** time and **O(V)** space.

In [ ]:
import io, gzip, time, requests
from collections import defaultdict
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

SEED = 42
SNAP_URL = "https://snap.stanford.edu/data/email-Eu-core.txt.gz"

r = requests.get(SNAP_URL, timeout=60)
r.raise_for_status()
with gzip.GzipFile(fileobj=io.BytesIO(r.content)) as gz:
    edges = pd.read_csv(gz, sep=" ", names=["source", "target"])

G = nx.from_pandas_edgelist(edges, "source", "target")
print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")
print(f"Connected components: {nx.number_connected_components(G)}")
print(f"Largest component size: {len(max(nx.connected_components(G), key=len)):,}")

## Custom iterative DFS

DFS goes **as deep as possible before backtracking**. It is ideal when the goal is exhaustive reachability or structural discovery rather than minimum-hop distance.

In [ ]:
def dfs_traversal(graph, start):
    if start not in graph:
        return [], {}

    stack = [(start, None, 0)]
    visited = set()
    parent = {}
    depth = {}
    order = []

    while stack:
        node, par, d = stack.pop()
        if node in visited:
            continue

        visited.add(node)
        parent[node] = par
        depth[node] = d
        order.append(node)

        # reverse gives deterministic-looking traversal for sorted neighbours
        for nbr in sorted(graph.neighbors(node), reverse=True):
            if nbr not in visited:
                stack.append((nbr, node, d + 1))

    return order, {"parent": parent, "depth": depth}

start = max(G.degree, key=lambda x: x[1])[0]
order, meta = dfs_traversal(G, start)

print("Start node:", start)
print("Reachable nodes discovered:", len(order))
print("Maximum DFS tree depth:", max(meta["depth"].values()))
print("First 20 visited nodes:", order[:20])

## Real example — investigate a deep relationship chain

Unlike BFS, the path in the DFS tree is **not guaranteed to be shortest**. That is useful to demonstrate in interviews because it shows you understand when to choose each search algorithm.

In [ ]:
target = order[min(250, len(order)-1)]

def reconstruct_path(parent, target):
    path = []
    cur = target
    while cur is not None:
        path.append(cur)
        cur = parent[cur]
    return path[::-1]

dfs_path = reconstruct_path(meta["parent"], target)
shortest = nx.shortest_path(G, start, target)

print("Target:", target)
print("DFS tree path length:", len(dfs_path)-1)
print("Actual shortest-path length:", len(shortest)-1)
print("DFS path is shortest:", len(dfs_path) == len(shortest))
print("DFS tree path:", " -> ".join(map(str, dfs_path[:20])) + (" ..." if len(dfs_path) > 20 else ""))

## Connected-component discovery with DFS

In [ ]:
def dfs_component(graph, seed, allowed=None):
    stack = [seed]
    visited = set()
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        if allowed is not None and node not in allowed:
            continue
        visited.add(node)
        for nbr in graph.neighbors(node):
            if nbr not in visited:
                stack.append(nbr)
    return visited

remaining = set(G.nodes())
component_sizes = []

while remaining:
    seed = next(iter(remaining))
    comp = dfs_component(G, seed, allowed=remaining)
    component_sizes.append(len(comp))
    remaining -= comp

component_sizes = sorted(component_sizes, reverse=True)
print("Top component sizes:", component_sizes[:10])
print("Component count from custom DFS:", len(component_sizes))
print("Matches NetworkX:", len(component_sizes) == nx.number_connected_components(G))

## Cycle detection

Cycles matter in dependency and infrastructure graphs because they can reveal recursive relationships or circular dependencies.

In [ ]:
def has_cycle_undirected(graph):
    visited = set()

    for start in graph.nodes():
        if start in visited:
            continue

        stack = [(start, None)]
        while stack:
            node, parent = stack.pop()

            if node in visited:
                continue
            visited.add(node)

            for nbr in graph.neighbors(node):
                if nbr == parent:
                    continue
                if nbr in visited:
                    return True
                stack.append((nbr, node))

    return False

print("Cycle detected:", has_cycle_undirected(G))

In [ ]:
# Visualise a manageable DFS tree slice
slice_nodes = order[:45]
H = G.subgraph(slice_nodes).copy()

tree_edges = []
for node in slice_nodes:
    par = meta["parent"].get(node)
    if par in H:
        tree_edges.append((par, node))

pos = nx.spring_layout(H, seed=SEED)

plt.figure(figsize=(12, 8))
nx.draw_networkx_edges(H, pos, alpha=0.12)
nx.draw_networkx_nodes(H, pos, node_size=120, alpha=0.75)
nx.draw_networkx_edges(H, pos, edgelist=tree_edges, width=2.2)
nx.draw_networkx_labels(H, pos, font_size=7)
plt.title("DFS traversal tree over a real email relationship graph")
plt.axis("off")
plt.show()

## DFS benchmark

In [ ]:
rng = np.random.default_rng(SEED)
starts = rng.choice(np.array(list(G.nodes())), size=100, replace=True)

latencies = []
reachable = []
for s in starts:
    t0 = time.perf_counter()
    visited, _ = dfs_traversal(G, int(s))
    latencies.append((time.perf_counter()-t0)*1000)
    reachable.append(len(visited))

print(f"Median full DFS traversal latency: {np.median(latencies):.3f} ms")
print(f"95th percentile latency: {np.percentile(latencies,95):.3f} ms")
print(f"Median reachable nodes: {np.median(reachable):,.0f}")

## Interview takeaway

**Use DFS when you need exhaustive depth-first exploration, connected components, cycle checks or dependency-chain discovery. Do not use it when you specifically need an unweighted shortest path.**

**CV bullet:** Built a DFS-based enterprise graph explorer on the real Stanford SNAP Email-Eu-core network, implementing iterative traversal, reachability, connected-component discovery, cycle detection, structural visualisation and latency benchmarking.